# Module 2 - Topic 3: Accessing LLMs via API
**Generative AI Fellowship — Beginner**

In this demo you will write your first Python code to talk to an LLM.

By the end of this notebook you will be able to:
- Make an API call to an LLM from Python
- Read the full response object
- Use system messages to shape the model's behaviour
- Build a multi-turn conversation
- Stream responses to the terminal
- Handle API errors


In [ ]:
# Cell 1 - Install
# Run this once to install the required libraries
# Remove the # to run it, then add it back after

# !pip install openai python-dotenv

In [ ]:
# Cell 2 - Import & Setup
# We import the tools we need, load our API key, and create the client

import os
from dotenv import load_dotenv
from openai import OpenAI

# This reads your .env file and loads the API key into memory
load_dotenv()

# This creates the client we will use to talk to the model
# The API key comes from your .env file — never hardcode it here
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("Client ready. Let's talk to an LLM.")

In [ ]:
# Cell 3 - Your First API Call
# We send a message to the model and print what it says back

response = client.chat.completions.create(
    model="gpt-4o-mini",       # Which model to use
    messages=[
        {
            "role": "user",     # This is our message
            "content": "What are the top three industries driving economic growth in Nigeria right now?"
        }
    ],
    temperature=0.7,            # How creative the response should be (0 = predictable, 1 = creative)
    max_tokens=300              # Maximum length of the response
)

# The actual text response is here
print(response.choices[0].message.content)

In [ ]:
# Cell 4 - Reading the Response Object
# The response contains more than just text
# Let's look at what else is inside

print("Text response:")
print(response.choices[0].message.content)

print("\nToken usage:")
print("  Prompt tokens:    ", response.usage.prompt_tokens)    # How many tokens we sent
print("  Completion tokens:", response.usage.completion_tokens) # How many tokens came back
print("  Total tokens:     ", response.usage.total_tokens)      # Total — this is what you pay for

print("\nFinish reason:", response.choices[0].finish_reason)
# 'stop'   = response ended naturally — good
# 'length' = response was cut off — increase max_tokens

In [ ]:
# Cell 5 - Adding a System Message
# The system message tells the model who it is and how to behave
# It shapes every response in the conversation

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",   # This sets the model's role — it reads this before answering
            "content": "You are a business advisor specialising in Nigerian startups. Always give practical, Nigeria-specific advice using Naira amounts and local examples."
        },
        {
            "role": "user",
            "content": "What are the top three industries driving economic growth in Nigeria right now?"
        }
    ],
    temperature=0.7,
    max_tokens=300
)

print(response.choices[0].message.content)

In [ ]:
# Cell 6 - Changing the System Message
# Same question — but watch how the response changes when we change the role

# Casual and motivating
response_casual = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a young Nigerian entrepreneur who just raised $1M in funding. Be casual, energetic and motivating."},
        {"role": "user", "content": "What are the top three industries driving economic growth in Nigeria right now?"}
    ],
    temperature=0.7,
    max_tokens=200
)

print("Casual entrepreneur role:")
print(response_casual.choices[0].message.content)

print("\n" + "-" * 50)

# Formal and analytical
response_formal = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a senior economist at the Central Bank of Nigeria. Be formal, data-driven and precise."},
        {"role": "user", "content": "What are the top three industries driving economic growth in Nigeria right now?"}
    ],
    temperature=0.7,
    max_tokens=200
)

print("\nCBN Economist role:")
print(response_formal.choices[0].message.content)

In [ ]:
# Cell 7 - Multi-Turn Conversation: The Broken Way
# LLMs have NO memory between calls
# Watch what happens when we ask a follow-up without including the history

# First question
response1 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "What is the best state in Nigeria to start a manufacturing business?"}
    ],
    max_tokens=150
)
print("First response:")
print(response1.choices[0].message.content)

print("\n" + "-" * 50)

# Follow-up WITHOUT history — the model has no idea what we discussed
response2 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Why did you recommend that specific state?"}
    ],
    max_tokens=150
)
print("\nFollow-up (broken — no history passed):")
print(response2.choices[0].message.content)

In [ ]:
# Cell 8 - Multi-Turn Conversation: The Right Way
# We must pass the full conversation history in every API call
# The 'assistant' role is how we include the model's previous responses

# Turn 1
turn1_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a Nigerian business development consultant."},
        {"role": "user", "content": "What is the best state in Nigeria to start a manufacturing business?"}
    ],
    max_tokens=150
)
turn1_text = turn1_response.choices[0].message.content
print("Turn 1:")
print(turn1_text)

print("\n" + "-" * 50)

# Turn 2 — we include the full history so the model remembers
turn2_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a Nigerian business development consultant."},
        {"role": "user", "content": "What is the best state in Nigeria to start a manufacturing business?"},
        {"role": "assistant", "content": turn1_text},   # The model's previous answer
        {"role": "user", "content": "Why did you recommend that specific state?"}  # Our follow-up
    ],
    max_tokens=150
)
print("\nTurn 2 (with full history — model remembers):")
print(turn2_response.choices[0].message.content)

In [ ]:
# Cell 9 - Streaming
# By default the API waits until the full response is ready before returning it
# Streaming sends each word as it is generated — much better for users

prompt = "Explain in 3 sentences why the Lagos tech ecosystem is growing rapidly."

print("Standard (waits for full response):")
print("-" * 50)
standard_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=200
)
print(standard_response.choices[0].message.content)

print("\n" + "=" * 50)
print("Streaming (text appears word by word):")
print("-" * 50)

stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=200,
    stream=True  # This enables streaming
)

# Each chunk contains one small piece of the response
for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)

print()  # New line after streaming finishes

In [ ]:
# Cell 10 - Error Handling
# API calls can fail — we need to handle errors gracefully
# The most common error for beginners: wrong API key

from openai import AuthenticationError, RateLimitError, APIError

# First — let's see what an error looks like with a wrong key
bad_client = OpenAI(api_key="sk-this-is-a-wrong-key")

try:
    bad_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": "Hello"}]
    )
except AuthenticationError:
    print("AuthenticationError: Invalid API key.")
    print("Fix: check your OPENAI_API_KEY in the .env file.")

print("\n" + "-" * 50)

# The safe way to make any API call
def ask_safely(prompt):
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=200
        )
        return response.choices[0].message.content

    except AuthenticationError:
        return "Error: Invalid API key. Check your .env file."

    except RateLimitError:
        return "Error: Too many requests. Please wait a moment and try again."

    except APIError as e:
        return f"Error: Something went wrong — {e}"

# Test it with a valid call
result = ask_safely("What is one piece of advice for a first-time Nigerian entrepreneur?")
print(result)